<a href="https://colab.research.google.com/github/M1ztick/SAIGE/blob/main/SAIGE_DPO_Training_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAIGE — DPO Fine-Tuning (v3)
**Model**: Qwen/Qwen2.5-3B-Instruct  
**Method**: Direct Preference Optimization (DPO) via TRL  
**Dataset**: M1ztyk/SAIGE-right-speech-dpo (85 pairs, prompt-diversified: rs/generic/none conditions)  
**Output**: M1ztyk/SAIGE-dpo-v3

**What changed from v2**: Hyperparameters only — dataset is unchanged, so v2 vs v3 isolates the training-config fix. v2's near-zero reward margins were a starved optimizer, not a data problem: `lr=5e-7` (a full-parameter DPO rate, too low for LoRA) combined with an effective batch of 16 over 68 pairs gave only **15 optimizer steps total**, ending at `train_loss=0.6829` against the zero-margin value of `ln(2)=0.6931`. v3 raises LR 10x to `5e-6` and cuts `grad_accum` 8→2 across 6 epochs for ~102 steps. `beta` stays at 0.1 — see the DPO Training cell for why lowering it would weaken learning rather than free it.

**Still outstanding (data quality, independent of this fix)**: `score_delta` is median 1 / mean 1.44 on the 10-point rubric, with 64 of 85 pairs at delta=1 — the chosen/rejected contrast is closer to "good vs. slightly better" than "right speech vs. wrong speech". Regenerating `rejected` responses to be genuinely poor (target delta ≥ 3) is the next lever after this run. Note that filtering alone won't do it: `diversify_prompts.py --min-delta 2` leaves only ~21 pairs.

**What changed in v2 (retained)**: Dataset stratifies system prompt conditions across three variants per record — RS prompt, generic prompt, and no system prompt. Teaches prompt-independent behavior rather than prompt-activated response.

In [ ]:
# Verify GPU
import subprocess
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print(result.stdout.strip())

Tesla T4, 15360 MiB


In [ ]:
# Install dependencies
#
# PINNED. Unpinned, Colab resolves to whatever shipped today, and two breaking
# changes have landed since the Sept 12 run this notebook benchmarks against:
#   - transformers merged `warmup_ratio` into a float `warmup_steps` (a value < 1
#     is now read as a fraction): deprecated in v5.0, removed in v5.2, so from
#     v5.2 on DPOConfig raises TypeError on `warmup_ratio=0.1`.
#   - TRL deprecated `rpo_alpha` in 0.28.0 and removed it in 0.29.0; the
#     replacement is `loss_type=["sigmoid", "sft"]` + `loss_weights`.
# trl 0.23.0 shipped 2025-09-10, two days before the run whose margins
# (0.171 -> 0.429) are the comparison point. Re-rolling the trainer library
# between runs would confound exactly the metric this notebook exists to measure.
#
# Restart the runtime after this cell so the pinned versions are the ones imported.
%pip install -q \
    "transformers<5" \
    "trl==0.23.0" \
    peft \
    accelerate \
    bitsandbytes \
    datasets \
    huggingface_hub

## Authentication
Token needs `write` scope. Get one at https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Load Dataset

In [ ]:
from datasets import load_dataset

# Filename matches the local source of truth (dpo_pairs_diversified.jsonl).
# v2 uploaded the diversified file to the hub *as* dpo_pairs.jsonl, which collided
# with the local pre-diversification dpo_pairs.jsonl — same name, different data.
# Re-run upload_dataset.py before training so this path exists on the hub.
dataset = load_dataset(
    "M1ztyk/SAIGE-right-speech-dpo",
    data_files="dpo_pairs_diversified.jsonl",
    split="train",
)
dataset = dataset.select_columns(["prompt", "chosen", "rejected"])

split = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train: {len(train_dataset)} pairs | Eval: {len(eval_dataset)} pairs")
print(f"Example prompt keys: {[m['role'] for m in train_dataset[0]['prompt']]}")

# Confirm the prompt-condition stratification actually survived the upload.
# If every row has a system message, you are training on the un-diversified file.
from collections import Counter
conditions = Counter(
    "none" if not any(m["role"] == "system" for m in r["prompt"])
    else ("rs" if "Right Speech" in next(m["content"] for m in r["prompt"] if m["role"] == "system")
          else "generic")
    for r in dataset
)
print(f"Prompt conditions: {dict(conditions)}  (expect all three present)")

dpo_pairs_diversified.jsonl:   0%|          | 0.00/314k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Train: 68 pairs | Eval: 17 pairs
Example prompt keys: ['system', 'user']
Prompt conditions: {'rs': 32, 'generic': 28, 'none': 25}  (expect all three present)


## Load Model + Tokenizer (QLoRA)
4-bit NF4 quantization via bitsandbytes — fits comfortably on T4 (16GB).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # T4 is Turing (CC 7.5) — BF16 requires Ampere+
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Configure tokenizer for DPO training length limits
# TRL >= 0.15.0 removed max_length/max_prompt_length from DPOTrainer/DPOConfig
tokenizer.model_max_length = 1024
tokenizer.truncation_side = "left"  # Truncate long prompts from the left

print(f"Model loaded. Params: {model.num_parameters() / 1e9:.2f}B")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model loaded. Params: 3.09B


## LoRA Configuration

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

## DPO Training

**What changed from v2 — the v2 run was effectively untrained.** v2 finished at `global_step=15` with `training_loss=0.6829`. The DPO loss at exactly zero preference margin is `ln(2) = 0.6931`, so 15 steps at `lr=5e-7` moved the loss 0.01 nats off its initialization value. The rising accuracy (0.61 → 0.78 → 0.89) said the preference signal was clean; the flat margin (0.0085 → 0.0197) said the weights barely moved.

Two independent causes, both fixed here:

- `lr=5e-7` → **`5e-6`**. 5e-7 is the standard full-parameter DPO rate. LoRA needs more: `lora_B` initializes to zero, so the adapter starts as a no-op and has to travel further to become anything. DPO-LoRA recipes typically run 5e-6 to 5e-5.
- `grad_accum=8` → **`2`**, `epochs=3` → **`6`**. At effective batch 16 over 68 train pairs, v2 got ~5 optimizer steps per epoch. Effective batch 4 over 6 epochs gives ~17/epoch ≈ **102 total steps** instead of 15.

`beta=0.1` is deliberately unchanged. Lowering it would be counterproductive here: the DPO gradient scales as `β·σ(−β·h)`, and at initialization `h≈0` so `σ(−βh)≈0.5` — meaning early gradient magnitude is proportional to β. Dropping beta to 0.01 would make early learning ~10× *weaker*, not freer. Beta caps how far the policy may drift at convergence; with v2's LR we never approached that cap, so beta was never the binding constraint.

Remaining hyperparameters:
- `beta=0.1` — KL leash on drift from the reference policy
- `ref_model=None` — TRL disables the LoRA adapter to get the reference forward pass, so no second model copy on the T4

**Reading the metrics**: TRL logs `rewards/chosen = beta * (policy_logps - ref_logps)`, so the reported margin is already multiplied by beta. At `beta=0.1`, a logged margin of 0.02 means a 0.2-nat log-prob gap. There is no beta-free target value — compare margins only across runs at the same beta.

**What to watch this run**: `rewards/margins` should climb well past 0.1, and `train_loss` should fall meaningfully below 0.69. Watch eval loss for overfitting — 6 epochs on 68 pairs is aggressive, and the fix for that is more/better pairs, not a smaller LR.

**Note for TRL >= 0.15.0**: `max_length` and `max_prompt_length` were removed from DPOTrainer/DPOConfig. Length limits are now handled via tokenizer configuration (see model loading cell above).

After training, run the 2x2 ablation in the inference notebook:
adapter+RS prompt / adapter+generic prompt / base+RS prompt / base+generic prompt.
The goal: adapter+generic should look close to adapter+RS.

---

**Run 3 changes (Sept 12-13).** The Sept 12 run reproduced the targets above - margins
0.171 -> 0.429, train_loss 0.2908 - confirming the v2 LR/step-count diagnosis. It was
then lost when the Colab VM recycled before the manual push cell ran. Four changes:

- `push_to_hub=True` + `hub_strategy="every_save"` - checkpoints now leave the VM as
  they are written, so a recycle costs minutes rather than the whole run.
- `epochs 6 -> 3` - 96% of the margin gain landed by epoch 4; epochs 5-6 added +0.004
  margin while train_loss fell a further 31%, which is overfitting, not learning.
- `rpo_alpha=1.0` - `rewards/chosen` reached -0.712, so the margin came from pushing
  rejected down rather than pulling chosen up. The NLL term counteracts that.
- `hub_model_id` -> `SAIGE-dpo-v3-run3` - the ablation loads `ADAPTER_ID` by bare repo
  name with no `revision=`, so pushing over `SAIGE-dpo-v3` would silently change what
  the Aug 3 ablation measured.

Note `rewards/accuracies` is not a clean accuracy: 17 eval pairs at batch size 2 give
9 batches averaged unweighted, so values quantise to 1/18 and the final single-example
batch counts double. Judge it by the 2x2 ablation, not this metric.


In [ ]:
from trl import DPOTrainer, DPOConfig
from peft import get_peft_model, prepare_model_for_kbit_training, PeftModel
import inspect
import torch

OUTPUT_DIR = "./saige-dpo-v3-output"
HUB_MODEL_ID = "M1ztyk/SAIGE-dpo-v3-run3"  # new repo: leaves the Aug 3 v3 adapter
                                           # intact as a comparison point

# 1. Strip any adapter left over from an earlier run in this session.
#    The previous `if not isinstance(model, PeftModel)` guard stopped double-wrapping
#    but silently WARM-STARTED: re-running this cell continued from the prior run's
#    weights instead of from zero, so "6 epochs" could really be 12 or 18.
if isinstance(model, PeftModel):
    model = model.unload()

# 2. Prepare for quantized training, then attach a fresh zero-init adapter
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

# 3. Aggressive cast to Float16 for all trainable parameters
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float16)

# Auto-detect available parameters
dpo_trainer_params = set(inspect.signature(DPOTrainer.__init__).parameters.keys())
dpo_config_params = set(inspect.signature(DPOConfig.__init__).parameters.keys())

# Build DPOConfig. Which arguments DPOConfig accepts moves with the transformers
# and TRL versions -- the install cell pins both, so nothing below should be
# dropped. Declare everything wanted and filter against the real signature anyway,
# so a future version bump prints a diff instead of raising halfway through the
# cell after the model is already on the GPU.
_desired = dict(
    output_dir=OUTPUT_DIR,
    hub_model_id=HUB_MODEL_ID,  # Trainer.push_to_hub()'s first positional arg is
                                # commit_message, not the repo — set the repo here
    num_train_epochs=3,  # 96% of the margin gain landed by epoch 4 last run;
                         # epochs 5-6 moved margins +0.004 while train_loss fell 31%
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,  # effective batch 4 → ~17 optimizer steps/epoch
    learning_rate=5e-6,  # 10x v2: LoRA needs more than full-param DPO's 5e-7
    beta=0.1,
    fp16=True,
    bf16=False,
    optim="paged_adamw_32bit",
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    report_to="none",
    remove_unused_columns=False,
    # Durability: the Sept 12 run was lost when the Colab VM recycled before the
    # manual push cell ran. Push each epoch's checkpoint as it completes instead.
    push_to_hub=True,
    hub_strategy="every_save",
    # Keep the best epoch by eval loss, not merely the last one.
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    max_length=1024,
    max_prompt_length=512,
    # RPO: adds an NLL term on the chosen completion. Last run drove rewards/chosen
    # to -0.712, i.e. the margin came from suppressing rejected rather than
    # preferring chosen. On TRL >= 0.29 this becomes loss_type=["sigmoid", "sft"]
    # with loss_weights=[1.0, rpo_alpha].
    rpo_alpha=1.0,
)

config_kwargs = {k: v for k, v in _desired.items() if k in dpo_config_params}
_dropped = sorted(set(_desired) - set(config_kwargs))
if _dropped:
    print(f"DPOConfig in this build does not accept: {_dropped}")

# Fail loudly rather than silently training on defaults - this is the class of bug
# that made v2 run 15 optimizer steps instead of 102.
_critical = {"learning_rate", "num_train_epochs", "per_device_train_batch_size",
             "gradient_accumulation_steps", "beta", "output_dir"}
_missing = _critical - set(config_kwargs)
if _missing:
    raise ValueError(f"DPOConfig rejected critical parameters: {sorted(_missing)}")

training_args = DPOConfig(**config_kwargs)

# Confirm what actually took effect, rather than what was requested.
for _k in ("learning_rate", "num_train_epochs", "beta", "warmup_ratio",
           "lr_scheduler_type", "rpo_alpha", "push_to_hub", "hub_strategy"):
    print(f"  {_k} = {getattr(training_args, _k, '<not supported>')}")

trainer_kwargs = dict(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

# Handle tokenizer naming convention in different TRL versions
if "processing_class" in dpo_trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = DPOTrainer(**trainer_kwargs)

# Workaround for the T4 BFloat16 GradScaler error: disable scaling
if trainer.accelerator.scaler is not None:
    trainer.accelerator.scaler._enabled = False

# Sanity check before burning GPU time: v2 silently ran only 15 steps.
steps_per_epoch = max(
    len(trainer.get_train_dataloader()) // training_args.gradient_accumulation_steps, 1
)
print(f"Planned optimizer steps: {steps_per_epoch}/epoch "
      f"x {training_args.num_train_epochs} epochs = "
      f"{steps_per_epoch * int(training_args.num_train_epochs)} total")
print(f"LR: {training_args.learning_rate} | beta: {training_args.beta}")
print("Zero-margin DPO loss is ln(2) = 0.6931 — train_loss should fall well below this.")
print(f"Pushing to https://huggingface.co/{HUB_MODEL_ID} at every epoch save.")
print("NOTE: needs a WRITE-scoped HF token; a 403 at epoch 1 would waste the run.")

trainer.train()

## Push to Hub

In [ ]:
# hub_strategy="every_save" already pushed each epoch, so the run survives a VM
# recycle. This cell is the belt-and-braces final artifact: save to disk first,
# then upload explicitly via HfApi (no commit_message/repo_id positional ambiguity,
# which is what sent v2 to M1ztyk/saige-dpo-v2-output).
trainer.model.save_pretrained("./v3_run3_final")
tokenizer.save_pretrained("./v3_run3_final")
print("Saved locally to ./v3_run3_final")

from huggingface_hub import HfApi
api = HfApi()
api.create_repo(HUB_MODEL_ID, exist_ok=True)
api.upload_folder(folder_path="./v3_run3_final", repo_id=HUB_MODEL_ID)

print(f"Done. Adapter at: https://huggingface.co/{HUB_MODEL_ID}")
print(f'Now set ADAPTER_ID = "{HUB_MODEL_ID}" in SAIGE_DPO_v3_Inference.ipynb '
      'before running the 2x2 ablation.')


README.md:   0%|          | 0.00/2.34k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpdpht1w9z/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Done. Adapter at: https://huggingface.co/M1ztyk/SAIGE-dpo-v3
Set ADAPTER_ID to this in SAIGE_DPO_Inference.ipynb before running the ablation.
